# SOCCAT — CV Pipeline (NLI Fine-Tuning)

Stratified K-fold cross-validation fine-tuning of `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` for binary social group mention detection.

**To replicate:** update the paths in the *Configuration* cell, then run all cells in order.

All 8 categories use the same pipeline — only the paths, epochs, and Hub repo differ. Those differences are documented in `categories.json`.


## 1. Installation

In [ ]:
%%capture
!pip install transformers datasets accelerate -U

## 2. Imports

In [ ]:
import argparse
import csv
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, average_precision_score, cohen_kappa_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from transformers import (
    AutoConfig, AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed,
)

## 3. Configuration

Update the paths and settings below. Everything else can be left at its default.

| Variable | Description |
|---|---|
| `PAIRS_PATH` | NLI pairs file (.csv or .xlsx) |
| `OUT_DIR` | Output directory for checkpoints and results |
| `EPOCHS` | Training epochs (3 for socio_economic_position and age_and_family_status; 4 for all others) |
| `PUSH_TO_HUB` | HF Hub repo id to push best model, or `None` to skip |

In [ ]:
# ── Paths (edit these) ────────────────────────────────────────────────────
PAIRS_PATH   = "data/nli_dataset_socio_economic_position.csv"
OUT_DIR      = "output/socio_economic_position"

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_NAME   = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# ── Hyperparameters ───────────────────────────────────────────────────────
SEED         = 42
K_FOLDS      = 5
MAX_LENGTH   = 256
EPOCHS       = 3       # 3 for socio_economic_position and age_and_family_status; 4 otherwise
LR           = 2e-5
BATCH_TRAIN  = 16
BATCH_EVAL   = 32
SEL_METRIC   = "f1_binary"

# ── Negative downsampling (off by default) ─────────────────────────────────
DOWNSAMPLE_NEG   = False
NEG_PER_POS_CAP  = 2

# ── Hub upload (set to None to skip) ──────────────────────────────────────
PUSH_TO_HUB  = None   # e.g. "selsar/cv_socio_economic_position"

# ── Reproducibility ───────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/cv_models", exist_ok=True)
os.makedirs(f"{OUT_DIR}/cv_results", exist_ok=True)
print("Configuration set.")

## 4. Data Loading

Accepts CSV or XLSX. Required columns: `sentence_id`, `premise`, `hypothesis`, `nli_label`, `hypothesis_label`, `outlet`, `country`, plus `year` or `date`.

In [ ]:
REQ_CORE = ["sentence_id", "premise", "hypothesis", "nli_label", "hypothesis_label"]
REQ_META = ["outlet", "country"]

def load_pairs(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path, engine="openpyxl")
    else:
        with open(path, "r", encoding="utf-8", newline="") as f:
            dialect = csv.Sniffer().sniff(f.read(4096), delimiters=",;\t|")
        df = pd.read_csv(path, delimiter=dialect.delimiter)

    missing = [c for c in REQ_CORE if c not in df.columns]
    if missing: raise ValueError(f"Missing required columns: {missing}")
    miss_meta = [c for c in REQ_META if c not in df.columns]
    if miss_meta: raise ValueError(f"Missing required metadata columns: {miss_meta}")
    if "year" not in df.columns and "date" not in df.columns:
        raise ValueError("Pairs file must contain 'year' or 'date'.")

    df["sentence_id"] = df["sentence_id"].astype(int)
    df["nli_label"]   = df["nli_label"].astype(int)

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        if "year" not in df.columns:
            df["year"] = df["date"].dt.year.astype("Int64")

    if "date" in df.columns:
        mask = df["date"].notna()
        df["decade"] = pd.NA
        if mask.any():
            df.loc[mask, "decade"] = (df.loc[mask, "date"].dt.year // 10 * 10).astype("Int64")
    else:
        df["decade"] = (pd.to_numeric(df["year"], errors="coerce") // 10 * 10).astype("Int64")
    return df


pairs = load_pairs(PAIRS_PATH)
labels = sorted(pairs["hypothesis_label"].dropna().unique().tolist())
with open(f"{OUT_DIR}/labels_found.json", "w") as f:
    json.dump(labels, f, indent=2)
print(f"Pairs loaded: {len(pairs):,} | Labels: {labels}")

## 5. Dataset Class

In [ ]:
class NLIDataset:
    """Premise-hypothesis NLI dataset for the HF Trainer."""
    def __init__(self, df, tokenizer, max_length):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["premise"], row["hypothesis"],
            truncation=True, max_length=self.max_length,
        )
        enc["labels"] = int(row["nli_label"])
        return enc

## 6. Stratified K-Fold

Iterative stratification assigns sentences to folds so that each positive label's frequency is balanced across folds (rare-first greedy).

In [ ]:
def _target_per_fold(count, k):
    base, rem = divmod(count, k)
    return [base + (1 if i < rem else 0) for i in range(k)]


def iterative_stratified_kfold(pos_long, labels, k, seed=42):
    rng = random.Random(seed)
    sents = sorted(pos_long["sentence_id"].unique())
    lab_map = {sid: set(pos_long.loc[pos_long.sentence_id == sid, "lab"]) for sid in sents}
    lab_counts = {L: int((pos_long["lab"] == L).sum()) for L in labels}
    target = {L: _target_per_fold(lab_counts[L], k) for L in labels}
    cur    = {L: [0] * k for L in labels}
    assign = {sid: None for sid in sents}
    remaining = set(sents)
    label_order = sorted(labels, key=lambda L: lab_counts[L])

    while remaining:
        best_def, best_lab, best_fold = -10**9, None, None
        for L in label_order:
            for f in range(k):
                deficit = target[L][f] - cur[L][f]
                if deficit > best_def:
                    best_def, best_lab, best_fold = deficit, L, f
        if best_def <= 0:
            for i, sid in enumerate(sorted(remaining)):
                assign[sid] = i % k
            break
        cands = [sid for sid in remaining if best_lab in lab_map[sid]]
        if not cands:
            cur[best_lab][best_fold] = target[best_lab][best_fold]
            continue
        def score(sid):
            return sum(1 for L in lab_map[sid] if (target[L][best_fold] - cur[L][best_fold]) > 0)
        scored = [(sid, score(sid)) for sid in cands]
        rng.shuffle(scored)
        best_sid = max(scored, key=lambda x: x[1])[0]
        assign[best_sid] = best_fold
        remaining.remove(best_sid)
        for L in lab_map[best_sid]:
            cur[L][best_fold] += 1

    return pd.DataFrame({"sentence_id": list(assign.keys()), "fold": list(assign.values())})


def export_fold_shares(pos_long, fold_map, out_csv):
    rows = []
    for f in sorted(fold_map["fold"].unique()):
        rep = (
            pos_long.merge(fold_map, on="sentence_id")
            .assign(split=lambda d: np.where(d["fold"] == f, "test", "train"))
            .groupby(["lab", "split"]).size().unstack(fill_value=0)
            .assign(
                total=lambda d: d.train + d.test,
                test_share=lambda d: np.where(d.total > 0, d.test / d.total, np.nan),
            )
            .reset_index().assign(fold=f)
        )
        rows.append(rep)
    pd.concat(rows, ignore_index=True).to_csv(out_csv, index=False)

## 7. Metrics

In [ ]:
def softmax(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def compute_all_metrics(y_true, prob_entail):
    """label 0 = entailment (positive), 1 = not_entailment."""
    y_pred   = np.where(prob_entail > 0.5, 0, 1)
    pos_mask = (y_true == 0).astype(int)
    out = dict(
        accuracy          = accuracy_score(y_true, y_pred),
        precision_binary  = precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        recall_binary     = recall_score(y_true, y_pred, pos_label=0),
        f1_binary         = f1_score(y_true, y_pred, pos_label=0),
        precision_micro   = precision_score(y_true, y_pred, average="micro", zero_division=0),
        recall_micro      = recall_score(y_true, y_pred, average="micro"),
        f1_micro          = f1_score(y_true, y_pred, average="micro"),
        f1_macro          = f1_score(y_true, y_pred, average="macro"),
        cohen_kappa       = cohen_kappa_score(y_true, y_pred),
        prevalence        = float(pos_mask.mean()),
    )
    try:    out["roc_auc"] = roc_auc_score(pos_mask, prob_entail)
    except: out["roc_auc"] = np.nan
    try:    out["pr_auc"]  = average_precision_score(pos_mask, prob_entail)
    except: out["pr_auc"]  = np.nan
    return out


def per_group_metrics(df_pairs, prob_entail, group_col):
    tmp = df_pairs.copy()
    tmp["prob_entail"] = prob_entail
    rows = []
    for g, sub in tmp.groupby(group_col, dropna=False):
        yt = sub["nli_label"].astype(int).values
        pe = sub["prob_entail"].values
        m  = compute_all_metrics(yt, pe)
        m.update({group_col: g, "n_pairs": len(sub), "n_pos_entail": int((yt == 0).sum())})
        rows.append(m)
    return pd.DataFrame(rows)


def add_time_fields(df):
    d = df.copy()
    if "year" not in d.columns and "date" in d.columns:
        d["date"] = pd.to_datetime(d["date"], errors="coerce")
        d["year"] = d["date"].dt.year.astype("Int64")
    if "decade" not in d.columns:
        if "date" in d.columns:
            mask = d["date"].notna()
            d["decade"] = pd.NA
            if mask.any():
                d.loc[mask, "decade"] = (d.loc[mask, "date"].dt.year // 10 * 10).astype("Int64")
        else:
            d["decade"] = (pd.to_numeric(d.get("year", pd.NA), errors="coerce") // 10 * 10).astype("Int64")
    return d

## 8. Cross-Validation Training Loop

Trains one model per fold, saves per-fold metrics, epoch loss tables, and per-group diagnostics (label, outlet, country, year, decade).

In [ ]:
def build_train_pairs(all_pairs, train_ids):
    train = all_pairs[all_pairs.sentence_id.isin(train_ids)].copy()
    if DOWNSAMPLE_NEG:
        keep = []
        for _, sub in train.groupby("sentence_id"):
            pos = sub[sub.nli_label == 0]
            neg = sub[sub.nli_label == 1]
            max_negs = NEG_PER_POS_CAP * max(1, len(pos))
            if len(neg) > max_negs:
                neg = neg.sample(n=max_negs, random_state=SEED)
            keep.append(pd.concat([pos, neg], ignore_index=True))
        if keep:
            train = pd.concat(keep, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return train


def build_test_pairs(all_pairs, test_ids):
    return all_pairs[all_pairs.sentence_id.isin(test_ids)].copy()


def compute_metrics_fn(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, (tuple, list)) else eval_pred.predictions
    prob_entail = softmax(logits)[:, 0]
    return compute_all_metrics(eval_pred.label_ids.astype(int), prob_entail)


# ── Stratification ────────────────────────────────────────────────────────
pos_long = (
    pairs[pairs.nli_label == 0][["sentence_id", "hypothesis_label"]]
    .drop_duplicates()
    .rename(columns={"hypothesis_label": "lab"})
)
if pos_long.empty:
    raise RuntimeError("No positive (entailment=0) rows; cannot stratify.")

fold_map = iterative_stratified_kfold(pos_long, labels, K_FOLDS, seed=SEED)
fold_map.to_csv(f"{OUT_DIR}/cv_sentence_folds.csv", index=False)
export_fold_shares(pos_long, fold_map, f"{OUT_DIR}/cv_results/stratification_shares.csv")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
epoch_tables, all_overall = [], []

for fold in range(K_FOLDS):
    print(f"\n========== Fold {fold + 1}/{K_FOLDS} ==========")
    test_ids  = set(fold_map.loc[fold_map.fold == fold, "sentence_id"])
    train_ids = set(pairs["sentence_id"]) - test_ids

    train_pairs = build_train_pairs(pairs, train_ids)
    test_pairs  = build_test_pairs(pairs, test_ids)

    ds_train = NLIDataset(train_pairs, tokenizer, MAX_LENGTH)
    ds_test  = NLIDataset(test_pairs,  tokenizer, MAX_LENGTH)

    config = AutoConfig.from_pretrained(
        MODEL_NAME, num_labels=2,
        id2label={0: "entailment", 1: "not_entailment"},
        label2id={"entailment": 0, "not_entailment": 1},
        problem_type="single_label_classification",
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, config=config, ignore_mismatched_sizes=True
    )

    training_args = TrainingArguments(
        output_dir                  = f"{OUT_DIR}/cv_models/fold_{fold}",
        learning_rate               = LR,
        per_device_train_batch_size = BATCH_TRAIN,
        per_device_eval_batch_size  = BATCH_EVAL,
        num_train_epochs            = EPOCHS,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = 1,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_accuracy",
        seed                        = SEED,
        report_to                   = [],
        logging_steps               = 50,
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = ds_train,
        eval_dataset    = ds_test,
        tokenizer       = tokenizer,
        data_collator   = DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics = compute_metrics_fn,
    )
    trainer.train()

    best_ckpt = trainer.state.best_model_checkpoint or trainer.args.output_dir

    # Epoch losses
    logs = pd.DataFrame(trainer.state.log_history)
    logs = logs[logs["epoch"].notna()].copy()
    tr = (logs[logs["loss"].notna()].groupby("epoch", as_index=False)["loss"].last()
          .rename(columns={"loss": "training_loss"})
          if "loss" in logs.columns else pd.DataFrame(columns=["epoch", "training_loss"]))
    ev = (logs[logs["eval_loss"].notna()].groupby("epoch", as_index=False)["eval_loss"].last()
          .rename(columns={"eval_loss": "validation_loss"})
          if "eval_loss" in logs.columns else pd.DataFrame(columns=["epoch", "validation_loss"]))
    epoch_tbl = pd.merge(tr, ev, on="epoch", how="outer").sort_values("epoch")
    epoch_tbl["fold"] = fold
    epoch_tbl.to_csv(f"{OUT_DIR}/cv_results/fold_{fold}_epoch_losses.csv", index=False)
    epoch_tables.append(epoch_tbl)

    # Held-out predictions
    pred        = trainer.predict(ds_test)
    logits      = pred.predictions[0] if isinstance(pred.predictions, (tuple, list)) else pred.predictions
    y_true      = pred.label_ids.astype(int)
    prob_entail = softmax(logits)[:, 0]
    y_pred      = np.where(prob_entail > 0.5, 0, 1)

    overall          = compute_all_metrics(y_true, prob_entail)
    overall["fold"]      = fold
    overall["best_ckpt"] = best_ckpt
    with open(f"{OUT_DIR}/cv_results/fold_{fold}_metrics.json", "w") as f:
        json.dump(overall, f, indent=2)
    all_overall.append(overall)

    # Per-group diagnostics
    tp = add_time_fields(test_pairs.copy())
    tp["prob_entail"] = prob_entail
    tp["pred_label"]  = y_pred

    def dump(df, name):
        df.to_csv(f"{OUT_DIR}/cv_results/fold_{fold}_per_{name}.csv", index=False)

    dump(per_group_metrics(tp, prob_entail, "hypothesis_label"), "label")
    dump(per_group_metrics(tp, prob_entail, "outlet"),           "outlet")
    dump(per_group_metrics(tp, prob_entail, "country"),          "country")
    dump(per_group_metrics(tp, prob_entail, "year"),             "year")
    dump(per_group_metrics(tp, prob_entail, "decade"),           "decade")

    output_cols = ["sentence_id", "premise", "hypothesis", "nli_label", "pred_label",
                   "prob_entail", "hypothesis_label", "outlet", "country", "year"]
    if "date" in tp.columns:
        output_cols.insert(6, "date")
    tp[output_cols].to_csv(f"{OUT_DIR}/cv_results/fold_{fold}_human_vs_model.csv", index=False)

print("\nAll folds complete.")

## 9. CV Summary & Best Model Selection

In [ ]:
metric_keys = ["accuracy", "precision_binary", "recall_binary", "f1_binary",
               "precision_micro", "recall_micro", "f1_micro", "f1_macro",
               "pr_auc", "roc_auc", "cohen_kappa"]
summary = {k: float(np.nanmean([m.get(k, np.nan) for m in all_overall])) for k in metric_keys}
with open(f"{OUT_DIR}/cv_results/summary_overall.json", "w") as f:
    json.dump(summary, f, indent=2)
print("==== CV SUMMARY ====")
for k, v in summary.items():
    print(f"  {k:25s}: {v:.4f}")

if epoch_tables:
    epochs_all = pd.concat(epoch_tables, ignore_index=True)
    epochs_all.rename(columns={"epoch": "Epoch", "training_loss": "Training Loss",
                                "validation_loss": "Validation Loss"}, inplace=True)
    epochs_all.to_csv(f"{OUT_DIR}/cv_results/epochs_all_folds.csv", index=False)

def _metric_val(m, key):
    v = m.get(key)
    return float("-inf") if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)

best_rec  = max(all_overall, key=lambda m: _metric_val(m, SEL_METRIC))
best_fold = int(best_rec["fold"])
best_ckpt = best_rec["best_ckpt"]
print(f"\nBest fold: {best_fold} | {SEL_METRIC} = {best_rec[SEL_METRIC]:.4f}")

best_dir = f"{OUT_DIR}/best_model"
os.makedirs(best_dir, exist_ok=True)
AutoModelForSequenceClassification.from_pretrained(best_ckpt).save_pretrained(best_dir)
AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True).save_pretrained(best_dir)

with open(os.path.join(best_dir, "cv_selection.json"), "w") as f:
    to_dump = {k: (float(v) if isinstance(v, (np.floating, float)) else v)
               for k, v in best_rec.items() if k != "best_ckpt"}
    to_dump.update({"selected_metric": SEL_METRIC, "best_checkpoint": best_ckpt})
    json.dump(to_dump, f, indent=2)
print(f"Best model saved to: {best_dir}")

## 10. (Optional) Push to Hugging Face Hub

Uncomment the second block or set `PUSH_TO_HUB` in the Configuration cell.

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import create_repo
    try:
        create_repo(PUSH_TO_HUB, exist_ok=True)
    except Exception as e:
        print(f"create_repo warning: {e}")
    final_model = AutoModelForSequenceClassification.from_pretrained(best_dir)
    final_tok   = AutoTokenizer.from_pretrained(best_dir)
    final_model.push_to_hub(PUSH_TO_HUB)
    final_tok.push_to_hub(PUSH_TO_HUB)
    print(f"Pushed to https://huggingface.co/{PUSH_TO_HUB}")
else:
    print("Hub upload skipped. Set PUSH_TO_HUB in the Configuration cell to enable.")